<a href="https://colab.research.google.com/github/jabri62018/Jabri_lab/blob/Jabri_lab/Zx_28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# file= Zx_28.ipynb
# Author= Eng. Abdulla Al-Jabri - مهندس عبدالله الجبري
# Independent Researcher
# Sana'a- Yemen
# jabri62018@gmail.com

import mpmath as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, zipfile, os
from google.colab import files

DPS = 80
mp.mp.dps = DPS
xp = 21.0

def Zx(t):
    x = mp.mpc('0.5', str(t))
    return mp.exp(-x/xp) * mp.exp(-5*mp.log(x)) * mp.log(x) * mp.sin(2*mp.pi/x)

def find_brackets(step=0.02, t_start=14.0, t_end=160, max_roots=28):
    mp.mp.dps = DPS - 20
    t_vals = np.arange(t_start, t_end, step)
    f_vals = [float(mp.im(Zx(t))) for t in t_vals]
    brackets = []
    for j in range(len(f_vals)-1):
        if not np.isnan(f_vals[j]) and not np.isnan(f_vals[j+1]):
            if f_vals[j] * f_vals[j+1] < 0:
                brackets.append((t_vals[j], t_vals[j+1]))
    return brackets[:max_roots]

def refine_roots(brackets, dps=DPS):
    mp.mp.dps = dps
    roots = []
    x_vals = []
    for t_min, t_max in brackets:
        root = mp.findroot(lambda tt: mp.im(Zx(tt)), (t_min, t_max),
                           tol=mp.mpf(f'1e-{dps-15}'), maxsteps=1500)
        if abs(float(mp.im(Zx(root)))) < 1e-50:
            roots.append(float(root))
            x_vals.append(t_min)
    return roots, x_vals

# 1. حساب الجذور
print("Searching for 28 zeros starting from gamma≈14...")
brackets = find_brackets()
roots, x_vals = refine_roots(brackets)
print(f"✓ Found {len(roots)} zeros. First gamma = {roots[0]:.12f}")

# 2. قائمة الـ28 ثابت
CONSTANTS = {
    'G': 6.67430e-11, 'alpha': 7.2973525693e-3, 'h': 6.62607015e-34,
    'c': 299792458.0, 'DE': 6.9e-27, 'LOCK': 1.0,
    'pi': np.pi, 'e': np.e, 'phi': 1.6180339887, 'zeta3': 1.202056903,
    'gamma_E': 0.5772156649, 'k_B': 1.380649e-23, 'N_A': 6.02214076e23,
    'mu0': 4*np.pi*1e-7, 'eps0': 8.854187817e-12, 'me': 9.10938356e-31,
    'mp': 1.67262192369e-27, 'ln2': np.log(2), 'ln10': np.log(10),
    'sqrt2': np.sqrt(2), 'sqrt3': np.sqrt(3), 'sqrt5': np.sqrt(5),
    'Catalan': 0.915965594, 'Glaisher': 1.282427129, 'Khinchin': 2.685452001,
    'Feigenbaum': 4.669201609, 'Mills': 1.3063778838, 'TwinPrime': 0.6601618158
}

MEANING = {
    'G':'Gravity', 'alpha':'Fine Structure', 'h':'Planck', 'c':'Light Speed',
    'DE':'Dark Energy', 'LOCK':'Lock Constant', 'pi':'Pi', 'e':'Euler',
    'phi':'Golden Ratio', 'zeta3':'Zeta(3)', 'gamma_E':'Euler-Mascheroni',
    'k_B':'Boltzmann', 'N_A':'Avogadro', 'mu0':'Vacuum Permeability',
    'eps0':'Vacuum Permittivity', 'me':'Electron Mass', 'mp':'Proton Mass',
    'ln2':'ln2', 'ln10':'ln10', 'sqrt2':'sqrt2', 'sqrt3':'sqrt3', 'sqrt5':'sqrt5',
    'Catalan':'Catalan', 'Glaisher':'Glaisher', 'Khinchin':'Khinchin',
    'Feigenbaum':'Feigenbaum', 'Mills':'Mills', 'TwinPrime':'Twin Prime'
}

# 3. حساب C_calc ومطابقة الثوابت
rows = []
available_consts = CONSTANTS.copy()
available_meaning = MEANING.copy()

for i, (g, x0) in enumerate(zip(roots, x_vals), 1):
    mp.mp.dps = DPS
    h = mp.mpf('1e-15')
    t = mp.mpf(g)
    z = Zx(t)
    zppp = (Zx(t+2*h) - 2*Zx(t+h) + 2*Zx(t-h) - Zx(t-2*h)) / (2*h)
    C_calc = 0.5 * g**2 * float(mp.re(zppp / z))

    logC = mp.log10(abs(C_calc) + mp.mpf('1e-300'))
    diffs = {k: abs(logC - mp.log10(abs(mp.mpf(v)) + mp.mpf('1e-300')))
             for k,v in available_consts.items()}
    matched = min(diffs, key=diffs.get)
    logdiff = float(diffs[matched])

    rows.append({
        'X': i,
        'x': x0,
        'gamma': float(g),
        'C_calc': float(C_calc),
        'Constant Value': CONSTANTS[matched],
        'symbol': matched,
        'Meaning': MEANING[matched],
        'Log Diff': logdiff
    })
    del available_consts[matched]
    del available_meaning[matched]

df = pd.DataFrame(rows)
df.to_csv('Zx28_roots_match.csv', index=False, float_format='%.15e')
print("Saved: Zx28_roots_match.csv")

# 4. الرسم
mp.mp.dps = DPS - 20
t_vals = np.linspace(14.0, 160, 8000)
f_vals = [float(mp.im(Zx(t))) for t in t_vals]

plt.figure(figsize=(14, 6))
plt.plot(t_vals, f_vals, linewidth=0.7, color='royalblue', label='Im[Zx(t)]')
plt.yscale('symlog', linthresh=1e-25)
plt.axhline(0, color='black', linewidth=1, alpha=0.6)

for i, row in df.iterrows():
    r = float(row['gamma'])
    plt.plot(r, 0, 'ro', markersize=6)
    plt.text(r, 1e-15, f"R{row['X']}-{row['symbol']}", ha='center', va='bottom',
             fontsize=8, color='darkred', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.7))

plt.xlabel('t', fontsize=12)
plt.ylabel('Im[Zx(t)] - Symlog Scale', fontsize=12)
plt.title(f'First 28 Zeros of Zx_28 Mapped to Physical Constants | {DPS}-digit precision',
          fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('Zx28_roots_plot.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: Zx28_roots_plot.png")

# 5. حفظ النوت بوك والضغط
nb = {
 "cells": [
  {"cell_type": "markdown",
   "source": ["# Zx_28 Root Analysis\n",
              "Computes first 28 zeros of Im[Zx(t)], matches each to a unique physical constant,",
              " and plots with symlog scale."]}
 ],
 "metadata": {"language_info": {"name": "python"}},
 "nbformat": 4, "nbformat_minor": 5
}
with open('Zx28_analysis.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=2)
print("Saved: Zx28_analysis.ipynb")

zip_name = 'Zx28_results.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('Zx28_roots_match.csv')
    zipf.write('Zx28_roots_plot.png')
    zipf.write('Zx28_analysis.ipynb')
print(f"\nDone! Created {zip_name}")
files.download(zip_name)

Searching for 28 zeros starting from gamma≈14...
✓ Found 3 zeros. First gamma = 15.053924744243
Saved: Zx28_roots_match.csv
Saved: Zx28_roots_plot.png
Saved: Zx28_analysis.ipynb

Done! Created Zx28_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>